In [ ]:
#hp_base_optimization_pipeline_path = "data\\hp_configm urations\\HyperparameterOptimizationPipelineLoaderDetach"
#hp_base_optimization_pipeline_path = "data\\hp_configurations\\HPOptDQNCartDetach"
#hp_base_optimization_pipeline_path = "data\\hp_configurations\\ppo_multi_hp_opt"
#hp_base_optimization_pipeline_path = "data\\hp_configurations\\ppo_multi_hp_opt_v2"
#hp_base_optimization_pipeline_path = "data\\hp_configurations\\ppo_par_multiagent_hp_opt"
#p_base_optimization_pipeline_path = "data\\hp_configurations\\ppo_multiwalker"
#hp_base_optimization_pipeline_path = "data\\hp_configurations\\ppo_jesp_multiwalker_1"
hp_base_optimization_pipeline_path = "data\\hp_configurations\\simple_spread_continuous_mappo_0"

In [ ]:
import shutil
from automarl.components.basic_components.artifact_management import open_or_create_folder
import os

base_experiment_path = "data\\hp_lodaded_exps"

if True : # if we copy the experiment path
    
    experiment_path = open_or_create_folder(base_experiment_path, "exp")

    shutil.copytree(hp_base_optimization_pipeline_path, experiment_path, dirs_exist_ok=True)

else:
    base_experiment_path = "data\\hp_other_exps"
    experiment_path = os.path.join(base_experiment_path, "multiwalker_ppo_1_125_1")

In [ ]:
from automarl.components.loggers.global_logger import activate_global_logger

activate_global_logger(experiment_path, {"necessary_logger_level" : "DEBUG"})

In [ ]:
from automarl.components.hp_opt.hp_optimization_pipeline import HyperparameterOptimizationPipeline
from automarl.utils.json_utils.json_component_utils import gen_component_from_path

hp_optimization_pipeline : HyperparameterOptimizationPipeline = gen_component_from_path(experiment_path)

hp_optimization_pipeline.pass_input({"logger_input" : {
    "necessary_logger_level" : "INFO",
    "write_to_file_when_text_lines_over" : -1
    }})

hp_optimization_pipeline.pass_input({"results_logger_input" : {
    "save_results_on_log" : True
}})


In [ ]:
from automarl.components.hp_opt.samplers.debug.tpe_debug import OptunaSamplerWrapperDebug


hp_optimization_pipeline.pass_input({"sampler" : (OptunaSamplerWrapperDebug, {
    "optuna_sampler" : "TreeParzen",
    "sampler_input": {
                    "n_startup_trials": 50,
                    "multivariate": True,
                    "group": True
                    }
        }
    )
    }
    )

In [ ]:
hp_optimization_pipeline.pass_input({"trainings_at_a_time" : 2})

In [ ]:
hp_optimization_pipeline.pass_input({"continue_after_error" : False})

In [ ]:
#hp_optimization_pipeline.pass_input({"steps" : 10})

In [ ]:
hp_optimization_pipeline.pass_input({"trainings_per_configuration" : 2})

In [ ]:
#hp_optimization_pipeline.pass_input({"continue_after_error" : False})

In [ ]:
hp_optimization_pipeline.pass_input({"do_initial_evaluation" : False})

In [ ]:
#hp_optimization_pipeline.pass_input({"start_with_given_values" : False})

In [ ]:
from automarl.components.loggers.debug.component_with_logging_debug import ComponentDebug
from automarl.components.ml.memory.debug.memory_debug import MemoryDebug
from automarl.components.ml.optimizers.debug.debug_optimizers import AdamOptimizerDebug
from automarl.components.rl.learners.debug.ppo_learner_debug import PPOLearnerDebug
from automarl.components.rl.policy.debug.policy_debug import PolicyDebug
from automarl.components.rl.policy.debug.policy_debug import QPolicyDebug
from automarl.components.rl.policy.debug.policy_debug import StochasticPolicyDebug
from automarl.components.rl.learners.debug.q_learner_debug import DQNLearnerDebug
from automarl.components.ml.models.debug.torch_model_debug import TorchModelComponentDebug
from automarl.components.rl.trainers.debug.agent_trainer_debug import AgentTrainerDebug
from automarl.components.rl.environment.pettingzoo.debug.parallel_petting_zoo_env import PettingZooEnvironmentWrapperParallelDebug
from automarl.components.rl.rl_player.debug.rl_player_debug import RLPlayerDebug
from automarl.components.rl.trainers.rl_trainer.debug.mappo_debug import RLTrainerMAPPODebug
from automarl.components.rl.learners.debug.ppo_learner_separated import PPOLearnerNoCriticDebug, PPOLearnerOnlyCriticDebug

if True:
  hp_optimization_pipeline.pass_input({"debug_classes":
                                     [
                                       #PolicyDebug,
                                       RLTrainerMAPPODebug,
                                       PPOLearnerNoCriticDebug,
                                       PPOLearnerOnlyCriticDebug,
                                       #PPOLearnerDebug,
                                       #TorchModelComponentDebug,
                                       #AdamOptimizerDebug,
                                       #QPolicyDebug,
                                       #StochasticPolicyDebug,
                                       #AgentTrainerDebug,
                                       #RLPlayerDebug,
                                       #MemoryDebug,
                                       #PettingZooEnvironmentWrapperParallelDebug,
                                       #ComponentDebug, # this should always be the last debug class, as the other extend it

                                     ]
                                     })

In [ ]:
hp_optimization_pipeline.process_input_if_not_processed()
hp_optimization_pipeline.run()

In [ ]:
from automarl.components.basic_components.state_management import save_state

save_state(hp_optimization_pipeline, save_definition=True)

In [ ]:
from automarl.components.hp_opt.hp_eval_results.hp_eval_results import print_optuna_trials_info

print_optuna_trials_info(hp_optimization_pipeline.get_study())